# 02_preprocessing

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/fake_job_postings.csv")

In [3]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Shape: (17880, 18)

Columns: ['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']


In [4]:
print("\nFirst look:")
df.head()


First look:


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [5]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
}).sort_values('Missing %', ascending=False)

print("\nMissing Values Summary:")
print(missing_df[missing_df['Missing Count'] > 0])


Missing Values Summary:
                     Missing Count  Missing %
salary_range                 15012      83.96
department                   11547      64.58
required_education            8105      45.33
benefits                      7212      40.34
required_experience           7050      39.43
function                      6455      36.10
industry                      4903      27.42
employment_type               3471      19.41
company_profile               3308      18.50
requirements                  2696      15.08
location                       346       1.94
description                      1       0.01


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   job_id               17880 non-null  int64 
 1   title                17880 non-null  object
 2   location             17534 non-null  object
 3   department           6333 non-null   object
 4   salary_range         2868 non-null   object
 5   company_profile      14572 non-null  object
 6   description          17879 non-null  object
 7   requirements         15184 non-null  object
 8   benefits             10668 non-null  object
 9   telecommuting        17880 non-null  int64 
 10  has_company_logo     17880 non-null  int64 
 11  has_questions        17880 non-null  int64 
 12  employment_type      14409 non-null  object
 13  required_experience  10830 non-null  object
 14  required_education   9775 non-null   object
 15  industry             12977 non-null  object
 16  func

# handle duplicate rows

In [7]:
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Dropped {before - len(df)} duplicate rows, new df shape: {df.shape}")

Dropped 0 duplicate rows, new df shape: (17880, 18)


# Handle Missing Values

In [8]:
before_rows = len(df)
df.dropna(subset=['description', 'location'], inplace=True)
dropped_rows = before_rows - len(df)
print(f"Dropped {dropped_rows} rows with missing 'description' or 'location'")

Dropped 347 rows with missing 'description' or 'location'


In [9]:
cat_cols = ['salary_range', 'required_experience', 'required_education', 'function', 'industry', 'employment_type']

for col in cat_cols:
    initial_nulls = df[col].isnull().sum()
    df[col] = df[col].fillna('Not Specified')
    if initial_nulls > 0:
        print(f"Filled {initial_nulls} nulls in '{col}' with 'Not Specified'")

initial_nulls_dept = df['department'].isnull().sum()
df['department'] = df['department'].fillna('Unknown')
if initial_nulls_dept > 0:
    print(f"Filled {initial_nulls_dept} nulls in 'department' with 'Unknown'")

text_cols = ['benefits', 'company_profile', 'requirements']

for col in text_cols:
    initial_nulls_text = df[col].isnull().sum()
    df[col] = df[col].fillna('')
    if initial_nulls_text > 0:
        print(f"Filled {initial_nulls_text} nulls in '{col}' with empty string")

Filled 14687 nulls in 'salary_range' with 'Not Specified'
Filled 6796 nulls in 'required_experience' with 'Not Specified'
Filled 7820 nulls in 'required_education' with 'Not Specified'
Filled 6192 nulls in 'function' with 'Not Specified'
Filled 4654 nulls in 'industry' with 'Not Specified'
Filled 3256 nulls in 'employment_type' with 'Not Specified'
Filled 11251 nulls in 'department' with 'Unknown'
Filled 6961 nulls in 'benefits' with empty string
Filled 3245 nulls in 'company_profile' with empty string
Filled 2519 nulls in 'requirements' with empty string


In [10]:
print("\n--- After Cleaning ---")
print("Shape:", df.shape)
print("\nRemaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])


--- After Cleaning ---
Shape: (17533, 18)

Remaining nulls:
Series([], dtype: int64)


In [11]:
print("\nClass distribution (fraudulent):")
print(df['fraudulent'].value_counts())
print(df['fraudulent'].value_counts(normalize=True).round(3) * 100)


Class distribution (fraudulent):
fraudulent
0    16687
1      846
Name: count, dtype: int64
fraudulent
0    95.2
1     4.8
Name: proportion, dtype: float64


# Drop Uninformative Columns

In [12]:
df = df.drop(columns=['job_id'])

In [13]:
print(df.shape)
df.sample(3)

(17533, 17)


,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
6504,Graduates: English Teacher Abroad (Conversatio...,"US, MN, Northfield",Unknown,Not Specified,We help teachers get safe &amp; secure jobs ab...,"Play with kids, get paid for it :-)Love travel...",University degree required. TEFL / TESOL / CEL...,See job description,0,1,1,Contract,Entry level,Bachelor's Degree,Education Management,Education,0
3874,Marketing Manager,"US, IL, Chicago",Unknown,45000-50000,Demo Duck builds handcrafted explainer videos ...,At Demo Duck we create handcrafted explainer v...,Talented writer &amp; editor - Your primary ro...,Healthcare stipend401K matchWork from home pol...,0,1,1,Full-time,Entry level,Bachelor's Degree,Marketing and Advertising,Marketing,0
14995,Project Engineer - Design Cost Engineer Exp - ...,"US, GA, Atlanta North",Unknown,Not Specified,We Provide Full Time Permanent Positions for m...,Experienced Project Engineer is required havin...,"4+ years’ experience as a cost engineer, proje...",,0,0,0,Full-time,Not Specified,Not Specified,Electrical/Electronic Manufacturing,Not Specified,0


#  Feature Type Organization

In [14]:
binary_features = ['telecommuting', 'has_company_logo', 'has_questions']
print("\n[Binary Features]")
for col in binary_features:
    unique_vals = df[col].unique()
    print(f"  {col}: {sorted(unique_vals)}")


categorical_features = ['employment_type', 'required_experience', 'required_education', 'industry', 'function']
print("\n[Categorical Features]")
for col in categorical_features:
    unique_vals = df[col].nunique()
    print(f"  {col}: {unique_vals} unique values → {df[col].unique()[:5]} ...")

text_features = ['title', 'company_profile', 'description', 'requirements', 'benefits']
print("\n[Text Features - Avg Length in Characters]")
for col in text_features:
    avg_len = df[col].astype(str).apply(len).mean()
    print(f"  {col}: {avg_len:.0f} chars avg")


[Binary Features]
  telecommuting: [np.int64(0), np.int64(1)]
  has_company_logo: [np.int64(0), np.int64(1)]
  has_questions: [np.int64(0), np.int64(1)]

[Categorical Features]
  employment_type: 6 unique values → ['Other' 'Full-time' 'Not Specified' 'Part-time' 'Contract'] ...
  required_experience: 8 unique values → ['Internship' 'Not Applicable' 'Not Specified' 'Mid-Senior level'
 'Associate'] ...
  required_education: 14 unique values → ['Not Specified' "Bachelor's Degree" "Master's Degree"
 'High School or equivalent' 'Unspecified'] ...
  industry: 132 unique values → ['Not Specified' 'Marketing and Advertising' 'Computer Software'
 'Hospital & Health Care' 'Online Media'] ...
  function: 38 unique values → ['Marketing' 'Customer Service' 'Not Specified' 'Sales'
 'Health Care Provider'] ...

[Text Features - Avg Length in Characters]
  title: 29 chars avg
  company_profile: 622 chars avg
  description: 1221 chars avg
  requirements: 595 chars avg
  benefits: 211 chars avg


In [15]:
df.to_csv('../data/processed/cleaned_data.csv', index=False)
print("\nCleaned data saved to: fake_job_postings_cleaned.csv")


Cleaned data saved to: fake_job_postings_cleaned.csv


In [16]:
# count missing values
missing = df.isnull().sum()

In [17]:
missing

title                  0
location               0
department             0
salary_range           0
company_profile        0
description            0
requirements           0
benefits               0
telecommuting          0
has_company_logo       0
has_questions          0
employment_type        0
required_experience    0
required_education     0
industry               0
function               0
fraudulent             0
dtype: int64